# 06. Prediction AUC

This notebook reproduces the local layer-13 predictive audit using the ten geometry features used in the March 2026 manuscript pass.
        


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))

from src.statistical_analysis import cohens_d, two_way_anova, stratified_logistic_auc

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 120)

PREDICTION_FEATURES = [
    'speed', 'dir_consistency', 'stabilization', 'turning_angle', 'dir_autocorr',
    'tortuosity', 'effective_dim', 'cos_slope', 'dist_slope', 'early_late_ratio'
]

df = pd.read_csv(DATA / 'qwen05b' / 'unified_metrics.csv')
layer_df = df[df['layer'] == 13].copy()
        


In [ ]:
rows = []
for condition in ['direct', 'cot']:
    sub = layer_df[layer_df['condition'] == condition].copy()
    valid = ~sub[PREDICTION_FEATURES].isna().any(axis=1)
    sub = sub[valid].copy()
    length_auc, length_std = stratified_logistic_auc(sub, ['response_length_chars'])
    geom_auc, geom_std = stratified_logistic_auc(sub, PREDICTION_FEATURES)
    combined_auc, combined_std = stratified_logistic_auc(sub, ['response_length_chars'] + PREDICTION_FEATURES)
    rows.extend([
        {'condition': condition, 'model': 'Length only', 'auc_mean': length_auc, 'auc_std': length_std},
        {'condition': condition, 'model': 'Geometry only', 'auc_mean': geom_auc, 'auc_std': geom_std},
        {'condition': condition, 'model': 'Combined', 'auc_mean': combined_auc, 'auc_std': combined_std},
    ])
auc_df = pd.DataFrame(rows)
auc_df
        


In [ ]:
plt.figure(figsize=(9, 5))
ax = sns.barplot(data=auc_df, x='condition', y='auc_mean', hue='model', palette='Set2')
for patch, (_, row) in zip(ax.patches, auc_df.iterrows()):
    ax.text(patch.get_x() + patch.get_width() / 2, row['auc_mean'] + 0.01, f"{row['auc_mean']:.3f}", ha='center', fontsize=9)
plt.ylim(0.5, 1.05)
plt.title('Layer-13 predictive AUC audit')
plt.ylabel('ROC AUC')
plt.tight_layout()
plt.savefig(FIGURES / 'repro_06_prediction_auc.png', dpi=300)
plt.show()
        
